# Modelo Preditivo — Regressão Linear para Nota de NPS

## 1. Definição do problema

O objetivo desta etapa é desenvolver um modelo de Regressão Linear Múltipla capaz de estimar a nota de satisfação (`nps_score`) dos clientes, utilizando informações relacionadas ao perfil, aos pedidos, às entregas e ao atendimento.

A proposta é antecipar a satisfação antes da aplicação da pesquisa de NPS, permitindo identificar clientes com maior risco de insatisfação e apoiar ações preventivas.

A previsão será considerada após a conclusão da experiência operacional, quando os indicadores utilizados pelo modelo já estiverem disponíveis, mas antes da coleta da nota de NPS.

## 2. Carregamento dos dados e importação das bibliotecas

In [12]:
import pandas as pd
import numpy as np

import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.linear_model import LinearRegression

In [3]:
dados = pd.read_csv("../data/raw/desafio_nps_fase_1.csv")

# Verificar dimensões e primeiras linhas
print(f"Quantidade de registros: {dados.shape[0]}")
print(f"Quantidade de colunas: {dados.shape[1]}")

dados.head()

Quantidade de registros: 2500
Quantidade de colunas: 19


,customer_id,customer_age,customer_region,customer_tenure_months,order_id,order_value,items_quantity,discount_value,payment_installments,delivery_time_days,delivery_delay_days,freight_value,delivery_attempts,customer_service_contacts,resolution_time_days,nps_score,repeat_purchase_30d,complaints_count,csat_internal_score
0,1,63,Nordeste,14,50001,139.73,4,39.35,4,2,2,55.53,3,0,4,6.9,0,3,6.5
1,2,20,Sul,1,50002,458.95,2,9.51,10,6,4,28.23,3,0,10,2.4,0,3,0.0
2,3,46,Nordeste,111,50003,507.06,5,42.82,6,6,1,40.99,1,4,5,4.8,0,7,1.5
3,4,52,Centro-Oeste,117,50004,302.19,2,19.58,9,5,2,35.24,3,1,11,5.9,0,4,0.3
4,5,56,Norte,50,50005,253.06,1,29.37,11,13,1,39.32,1,1,0,6.1,0,3,7.9


## 3. Seleção das Variáveis

Nosso taget (`y`) será a variável `nps_score`, representando a nota individual de satisfação do cliente.

As variáveis preditoras (`X`) serão selecionadas entre as informações disponíveis antes da pesquisa de NPS, não incluindo variáveis que sofreriam de data leakage.

A seleção considera os resultados da análise exploratória, mas não será limitada às variáveis com maior correlação, pois outras características podem contribuir para a capacidade preditiva do modelo.

In [5]:
# Preditoras
variaveis_modelo = [
    "customer_age",
    "customer_region",
    "customer_tenure_months",
    "order_value",
    "items_quantity",
    "discount_value",
    "payment_installments",
    "delivery_time_days",
    "delivery_delay_days",
    "freight_value",
    "delivery_attempts",
    "customer_service_contacts",
    "resolution_time_days",
    "complaints_count"
]

# Preditoras (X) e Target (y)
X = dados[variaveis_modelo].copy()
y = dados["nps_score"].copy()

print(f"Variáveis preditoras: {X.shape[1]}")
print(f"Registros: {X.shape[0]}")
print(f"Target: {y.name}")

Variáveis preditoras: 14
Registros: 2500
Target: nps_score


Variáveis que foram deixadas de fora:

- customer_id e order_id: identificadores.
- repeat_purchase_30d e csat_internal_score: informações que podem ser coletadas após pesquisa de NPS (data leakage).

## 4. Preparação dos Dados

Antes de treinar o modelo, vamos verificar os tipos das variáveis preditoras e a existência de valores ausentes, identificando quais transformações serão necessárias para a regressão linear.

In [6]:
print("Tipos das variáveis:")
print(X.dtypes)

# Verificação de valores ausentes
print("\nValores ausentes:")
print(X.isnull().sum())
print("\nValores ausentes no target:", y.isnull().sum())

Tipos das variáveis:
customer_age                   int64
customer_region                  str
customer_tenure_months         int64
order_value                  float64
items_quantity                 int64
discount_value               float64
payment_installments           int64
delivery_time_days             int64
delivery_delay_days            int64
freight_value                float64
delivery_attempts              int64
customer_service_contacts      int64
resolution_time_days           int64
complaints_count               int64
dtype: object

Valores ausentes:
customer_age                 0
customer_region              0
customer_tenure_months       0
order_value                  0
items_quantity               0
discount_value               0
payment_installments         0
delivery_time_days           0
delivery_delay_days          0
freight_value                0
delivery_attempts            0
customer_service_contacts    0
resolution_time_days         0
complaints_count         

### 4.1 Tratamento de Variáveis Categóricas

A base não possui valores ausentes, mas a variável `customer_region` é categórica.

Será utilizado One-Hot Encoding para transformar as regiões em variáveis numéricas, sem estabelecer uma ordem artificial entre elas.

In [8]:
X_treino, X_teste, y_treino, y_teste = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42
)

print(f"Treino: {X_treino.shape[0]} registros")
print(f"Teste: {X_teste.shape[0]} registros")

Treino: 2000 registros
Teste: 500 registros


In [10]:
# Variável categórica
variaveis_categoricas = ["customer_region"]

# Configurar transformação
preprocessador = ColumnTransformer(
    transformers=[
        (
            "regiao",
            OneHotEncoder(drop="first", handle_unknown="ignore"),
            variaveis_categoricas
        )
    ],
    remainder="passthrough",
    verbose_feature_names_out=False
)

# Aprender as categorias usando somente o treino
X_treino_transformado = preprocessador.fit_transform(X_treino)

# Aplicar a mesma transformação ao teste
X_teste_transformado = preprocessador.transform(X_teste)

print(f"Treino transformado: {X_treino_transformado.shape}")
print(f"Teste transformado: {X_teste_transformado.shape}")

Treino transformado: (2000, 17)
Teste transformado: (500, 17)


In [ ]:
# Resultado em DataFrame para visualização
X_treino_visualizacao = pd.DataFrame(
    X_treino_transformado,
    columns=preprocessador.get_feature_names_out(),
    index=X_treino.index
)

display(X_treino_visualizacao.head())

,customer_region_Nordeste,customer_region_Norte,customer_region_Sudeste,customer_region_Sul,customer_age,customer_tenure_months,order_value,items_quantity,discount_value,payment_installments,delivery_time_days,delivery_delay_days,freight_value,delivery_attempts,customer_service_contacts,resolution_time_days,complaints_count
2055,1.0,0.0,0.0,0.0,67.0,23.0,604.69,6.0,7.52,3.0,5.0,3.0,41.70,2.0,2.0,10.0,4.0
1961,0.0,0.0,0.0,1.0,66.0,13.0,162.00,5.0,20.57,7.0,2.0,2.0,58.27,1.0,2.0,4.0,4.0
1864,0.0,0.0,0.0,1.0,22.0,24.0,164.79,5.0,50.78,9.0,6.0,2.0,47.99,3.0,1.0,0.0,5.0
2326,1.0,0.0,0.0,0.0,52.0,93.0,254.79,6.0,96.14,6.0,8.0,1.0,52.02,1.0,0.0,11.0,3.0
461,0.0,0.0,0.0,1.0,57.0,48.0,222.50,3.0,48.65,6.0,9.0,5.0,48.83,2.0,6.0,3.0,10.0


## 5. Treinamento do Modelo

In [14]:
modelo = LinearRegression()

modelo.fit(X_treino_transformado, y_treino)

,"fit_intercept fit_intercept: bool, default=TrueWhether to calculate the intercept for this model. If setto False, no intercept will be used in calculations(i.e. data is expected to be centered).",True
,"copy_X copy_X: bool, default=TrueIf True, X will be copied; else, it may be overwritten.",True
,"tol tol: float, default=1e-6The precision of the solution (`coef_`) is determined by `tol` whichspecifies the convergence criterion of the underlying solver. `tol` isset as `atol` and `btol` of :func:`scipy.sparse.linalg.lsqr` whenfitting on sparse training data. `tol` is set as `cond` of:func:`scipy.linalg.lstsq` when fitting on dense training data... versionadded:: 1.7.. versionchanged:: 1.9 Now supported on dense data, interpreted as the `cond` parameter.",1e-06
,"n_jobs n_jobs: int, default=NoneThe number of jobs to use for the computation. This will only providespeedup in case of sufficiently large problems, that is if firstly`n_targets > 1` and secondly `X` is sparse or if `positive` is setto `True`. ``None`` means 1 unless in a:obj:`joblib.parallel_backend` context. ``-1`` means using allprocessors. See :term:`Glossary <n_jobs>` for more details.",None
,"positive positive: bool, default=FalseWhen set to ``True``, forces the coefficients to be positive. Thisoption is only supported for dense arrays.For a comparison between a linear regression model with positive constraintson the regression coefficients and a linear regression without such constraints,see :ref:`sphx_glr_auto_examples_linear_model_plot_nnls.py`... versionadded:: 0.24",False
Name,Type,Value
"coef_ coef_: array of shape (n_features, ) or (n_targets, n_features)Estimated coefficients for the linear regression problem.If multiple targets are passed during the fit (y 2D), thisis a 2D array of shape (n_targets, n_features), while if onlyone target is passed, this is a 1D array of length n_features.","ndarray[float64](17,)","[ 0.12, 0.17, 0.13,...,-0.34,-0.15,-0.38]"
"intercept_ intercept_: float or array of shape (n_targets,)Independent term in the linear model. Set to 0.0 if`fit_intercept = False`.",float64,9.089
n_features_in_ n_features_in_: intNumber of features seen during :term:`fit`... versionadded:: 0.24,int,17
rank_ rank_: intRank of matrix `X`. Only available when `X` is dense.,int64,np.int64(17)
"singular_ singular_: array of shape (min(X, y),)Singular values of `X`. Only available when `X` is dense.","ndarray[float64](17,)","[12889. , 1536.98, 1317.13,..., 20.09, 19.62, 8.75]"
